# Week 50, Day 6: Forge: Week 50 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w050/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(2 * 110 + 3 * np.log(200), 1))
print("2.", round(np.exp(4.6), 0))
print("3.", round(1 / (1 - 0.98)), "trials")

### Exercise: Part A: cross-validating the number of states

Using the provided forward_backward and fit_glmhmm, fit K = 1, 2 and 3 to the training trials and store the mean log-likelihood per test trial, computed from the scaling constants of a forward pass on the test trials.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2022)
T = 3000
W_true = np.array([[4.0, 0.0], [0.5, 1.5]]); A_true = np.array([[0.98, 0.02], [0.03, 0.97]])
s = np.zeros(T, int)
for t in range(1, T):
    s[t] = (1 - s[t - 1]) if r.random() > A_true[s[t - 1], s[t - 1]] else s[t - 1]
stim = r.choice([-1, -0.5, -0.25, -0.125, 0, 0.125, 0.25, 0.5, 1], T)
X = np.c_[stim, np.ones(T)]
y = (r.random(T) < 1 / (1 + np.exp(-(X * W_true[s]).sum(1)))).astype(float)

def forward_backward(X, y, W, A, pi):
    P = 1 / (1 + np.exp(-X @ W.T)); like = np.where(y[:, None] == 1, P, 1 - P); T_, K = like.shape
    al = np.zeros((T_, K)); c = np.zeros(T_); al[0] = pi * like[0]; c[0] = al[0].sum(); al[0] /= c[0]
    for t in range(1, T_):
        al[t] = (al[t - 1] @ A) * like[t]; c[t] = al[t].sum(); al[t] /= c[t]
    be = np.ones((T_, K))
    for t in range(T_ - 2, -1, -1):
        be[t] = (A @ (like[t + 1] * be[t + 1])) / c[t + 1]
    g = al * be; g /= g.sum(1, keepdims=True)
    return g, c, al, be, like

def fit_glmhmm(X, y, K, iters=25, seed=0):
    g0 = np.random.default_rng(seed)
    W = g0.normal(0, 1, (K, 2)); A = np.full((K, K), 0.05 / max(K - 1, 1)); np.fill_diagonal(A, 0.95 if K > 1 else 1.0); pi = np.ones(K) / K
    for it in range(iters):
        g, c, al, be, like = forward_backward(X, y, W, A, pi)
        if K > 1:
            xi = np.zeros((K, K))
            for t in range(len(y) - 1):
                xi += al[t][:, None] * A * (like[t + 1] * be[t + 1])[None] / c[t + 1]
            A = xi / xi.sum(1, keepdims=True)
        pi = g[0]
        for k in range(K):
            for _ in range(5):
                pk = 1 / (1 + np.exp(-X @ W[k])); grad = X.T @ (g[:, k] * (y - pk))
                H = -(X * (g[:, k] * pk * (1 - pk))[:, None]).T @ X - 1e-3 * np.eye(2)
                W[k] -= np.linalg.solve(H, grad)
    return W, A, pi

train, test = slice(0, 2000), slice(2000, 3000)
held = {}
for K in [1, 2, 3]:
    W, A, pi = fit_glmhmm(X[train], y[train], K)
    ...
    print(f"K = {K}: held-out log-likelihood per trial {held[K]:.4f}")

In [ ]:
import numpy as np
def test_two_beats_one():
    "two states improve held-out log-likelihood per trial by more than 0.05 over one"
    assert held[2] - held[1] > 0.05
def test_three_no_gain():
    "a third state adds less than 0.01 per trial"
    assert held[3] - held[2] < 0.01

test_two_beats_one()
test_three_no_gain()
print("All checks passed")

### Exercise: Part B: from choices to population frequencies

Write choice_prob and nll as in Day 1, then for each participant fit the three models with L-BFGS-B, store -BIC/2 in lme, and run rfx_bms. Count participants whose highest-evidence model is their true one.

In [ ]:
from scipy.optimize import minimize
from scipy.special import digamma
T2 = 200; mu = np.array([0.2, 0.8])

def choice_prob(model, par, Q, t, a, rw):
    if model == "ql":
        return 1 / (1 + np.exp(-par[1] * (Q[1] - Q[0])))
    if model == "wsls":
        if t == 0:
            return 0.5
        stay = 1 - par[0] / 2
        return stay if ((a[t - 1] == 1) == (rw[t - 1] == 1)) else 1 - stay
    return par[0]

def simulate(model, par):
    a, rw, Q = np.zeros(T2, int), np.zeros(T2), np.zeros(2)
    for t in range(T2):
        a[t] = r.random() < choice_prob(model, par, Q, t, a, rw)
        rw[t] = r.random() < (mu if t < T2 // 2 else mu[::-1])[a[t]]
        if model == "ql":
            Q[a[t]] += par[0] * (rw[t] - Q[a[t]])
    return a, rw

def nll(model, par, a, rw):
    Q = np.zeros(2); L = 0.0
    for t in range(T2):
        p1 = min(max(choice_prob(model, par, Q, t, a, rw), 1e-6), 1 - 1e-6)
        L -= np.log(p1 if a[t] == 1 else 1 - p1)
        if model == "ql":
            Q[a[t]] += par[0] * (rw[t] - Q[a[t]])
    return L

def rfx_bms(lme, iters=100):
    a0 = np.ones(lme.shape[1]); a = a0.copy()
    for _ in range(iters):
        lu = lme + digamma(a) - digamma(a.sum()); lu -= lu.max(1, keepdims=True)
        u = np.exp(lu); g = u / u.sum(1, keepdims=True); a = a0 + g.sum(0)
    return a, g

spec = {"ql": ([0.5, 3.0], [(0.01, 1), (0, 20)]), "wsls": ([0.3], [(0.01, 1)]), "bias": ([0.5], [(0.01, 0.99)])}
names = list(spec)
truth = ["ql"] * 14 + ["wsls"] * 6
lme = np.zeros((20, 3))
for i, m in enumerate(truth):
    par = [r.uniform(0.1, 0.6), r.uniform(2, 8)] if m == "ql" else [r.uniform(0.05, 0.3)]
    a_, rw_ = simulate(m, par)
    for j, (mod, (x0, bounds)) in enumerate(spec.items()):
        res = minimize(lambda p: nll(mod, p, a_, rw_), x0, method="L-BFGS-B", bounds=bounds)
        lme[i, j] = ...
alpha, g = rfx_bms(lme)
freq = alpha / alpha.sum()
correct = sum(names[np.argmax(lme[i])] == truth[i] for i in range(20))
print(f"participants assigned to their true model: {correct} of 20")
print("expected frequencies:", {n: round(float(f), 3) for n, f in zip(names, freq)})

In [ ]:
import numpy as np
def test_assign():
    "at least 17 of 20 participants are assigned to their true model"
    assert correct >= 17
def test_freq():
    "the expected frequency of Q-learning is between 0.55 and 0.8"
    assert 0.55 < freq[0] < 0.8

test_assign()
test_freq()
print("All checks passed")